# SMART Rollout Simulation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart_rollout_simulation_colab.ipynb)

This notebook is the **simulation stage** only:
- load baseline/variant checkpoints,
- optionally run SMART validation commands,
- write per-model simulation manifests for strict downstream provenance checks.


In [ ]:
# Step 1: Sync repo + deterministic Colab bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-constrained config + simulation contract context
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

def _bool_env(name: str, default: bool) -> bool:
    val = os.environ.get(name, "").strip().lower()
    if not val:
        return bool(default)
    return val in {"1", "true", "yes", "y", "on"}

def _latest_dir(root: Path, pattern: str) -> Path | None:
    if not root.exists():
        return None
    paths = [p for p in root.glob(pattern) if p.exists() and p.is_dir()]
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)
outputs_root = persist_run_dir / "outputs"
outputs_root.mkdir(parents=True, exist_ok=True)

AUTO_RESUME = _bool_env("WOSAC_AUTO_RESUME", True)
explicit_run_tag = os.environ.get("WOSAC_RUN_TAG", "").strip()
if explicit_run_tag:
    RUN_TAG = explicit_run_tag
elif AUTO_RESUME:
    latest = _latest_dir(outputs_root, "*")
    RUN_TAG = latest.name if latest is not None else datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
else:
    RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RUN_OUTPUT_DIR = outputs_root / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

latest_sim_dir = _latest_dir(outputs_root, "*/simulation_manifests") if AUTO_RESUME else None
latest_rollout_dir = _latest_dir(outputs_root, "*/rollout_protos") if AUTO_RESUME else None

SIM_MANIFESTS_DIR = Path(
    os.environ.get("WOSAC_SIM_MANIFESTS_DIR", str(latest_sim_dir or (RUN_OUTPUT_DIR / "simulation_manifests")))
).expanduser()
ROLLOUT_PROTO_DIR = Path(
    os.environ.get("WOSAC_ROLLOUT_PROTO_DIR", str(latest_rollout_dir or (RUN_OUTPUT_DIR / "rollout_protos")))
).expanduser()
SCENARIO_PROTO_DIR = os.environ.get("WOSAC_SCENARIO_PROTO_DIR", "").strip()
SCENARIO_PROTO_PATH = os.environ.get("WOSAC_SCENARIO_PROTO_PATH", "").strip()
SCENARIO_TFRECORDS = os.environ.get("WOSAC_SCENARIO_TFRECORDS", "").strip()

SCENARIO_SET_ID = os.environ.get("WOSAC_SCENARIO_SET_ID", "womd_validation").strip()
SCENARIO_SET_HASH = os.environ.get("WOSAC_SCENARIO_SET_HASH", "").strip()
EVALUATOR_ID = os.environ.get("WOSAC_EVALUATOR_ID", "waymo_open_dataset.sim_agents_metrics.challenge_2025").strip()
METRICS_CONFIG_HASH = os.environ.get("WOSAC_METRICS_CONFIG_HASH", "").strip()
SIM_SEED = int(os.environ.get("WOSAC_SIM_SEED", "2"))

print("RUN_TAG:", RUN_TAG)
print("AUTO_RESUME:", AUTO_RESUME)
print("persist_run_dir:", persist_run_dir)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("SIM_MANIFESTS_DIR:", SIM_MANIFESTS_DIR)
print("ROLLOUT_PROTO_DIR:", ROLLOUT_PROTO_DIR)
print("SCENARIO_PROTO_DIR:", SCENARIO_PROTO_DIR or "<set if using per-scenario proto files>")
print("SCENARIO_PROTO_PATH:", SCENARIO_PROTO_PATH or "<optional single-scenario proto>")
print("SCENARIO_TFRECORDS:", SCENARIO_TFRECORDS or "<optional tfrecord fallback>")
print("SCENARIO_SET_ID:", SCENARIO_SET_ID)
print("SCENARIO_SET_HASH:", SCENARIO_SET_HASH or "<set this for strict provenance>")
print("EVALUATOR_ID:", EVALUATOR_ID)
print("METRICS_CONFIG_HASH:", METRICS_CONFIG_HASH or "<set this for strict provenance>")
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Define checkpoints to simulate + rollout proto outputs (auto-discovery friendly)
from pathlib import Path

def _latest_ckpt(search_root: Path) -> str:
    if (not search_root.exists()) or (not search_root.is_dir()):
        return ""
    ckpts = [p for p in search_root.rglob("*.ckpt") if p.is_file()]
    if not ckpts:
        return ""
    ckpts.sort(key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    return str(ckpts[0])

def _latest_json(root: Path, pattern: str) -> Path | None:
    if not root.exists():
        return None
    paths = [p for p in root.glob(pattern) if p.exists() and p.is_file()]
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

baseline_ckpt = os.environ.get("SMART_BASELINE_CKPT", "").strip()
baseline_ckpt_root = Path(
    os.environ.get(
        "SMART_BASELINE_CKPT_ROOT",
        str(Path(PERSIST_ROOT) / f"smart_baseline_{RUN_NAME}" / "checkpoints"),
    )
).expanduser()
if not baseline_ckpt:
    baseline_ckpt = _latest_ckpt(baseline_ckpt_root)
assert baseline_ckpt, "Could not auto-discover baseline checkpoint. Set SMART_BASELINE_CKPT."

variant_ckpts = [v.strip() for v in os.environ.get("SMART_VARIANT_CKPTS", "").split(",") if v.strip()]
max_variants = int(os.environ.get("WOSAC_MAX_VARIANTS", "3"))
variant_filter = {v.strip() for v in os.environ.get("WOSAC_VARIANT_FILTER", "").split(",") if v.strip()}

if not variant_ckpts:
    latest_constrained = _latest_json(
        Path(PERSIST_ROOT) / f"smart_constrained_{RUN_NAME}" / "outputs",
        "*/smart_constrained_train_v0.json",
    )
    if latest_constrained is not None:
        payload = json.loads(latest_constrained.read_text(encoding="utf-8"))
        scan = dict(payload.get("variant_ckpt_scan", {}) or {})
        discovered = []
        for variant_id, row in scan.items():
            if variant_filter and variant_id not in variant_filter:
                continue
            ckpt_files = [str(x).strip() for x in (row.get("checkpoint_files", []) or []) if str(x).strip()]
            if ckpt_files:
                discovered.append(ckpt_files[-1])
                continue
            ckpt_dir = Path(str(row.get("checkpoint_dir", "")).strip()).expanduser()
            latest = _latest_ckpt(ckpt_dir)
            if latest:
                discovered.append(latest)
        variant_ckpts = discovered[: max(0, max_variants)]

baseline_rollouts = os.environ.get("SMART_BASELINE_ROLLOUTS_PROTO", "").strip() or str(ROLLOUT_PROTO_DIR / "smart_baseline.binproto")
variant_rollouts = [v.strip() for v in os.environ.get("SMART_VARIANT_ROLLOUTS_PROTOS", "").split(",") if v.strip()]

models = [
    {
        "model_id": "smart_baseline",
        "checkpoint_path": baseline_ckpt,
        "scenario_rollouts_path": baseline_rollouts,
        "env": {},
    }
]
for idx, ckpt in enumerate(variant_ckpts, start=1):
    rollout_path = variant_rollouts[idx - 1] if idx - 1 < len(variant_rollouts) else str(ROLLOUT_PROTO_DIR / f"variant_{idx}.binproto")
    models.append(
        {
            "model_id": f"variant_{idx}",
            "checkpoint_path": ckpt,
            "scenario_rollouts_path": rollout_path,
            "env": {},
        }
    )

print("baseline_ckpt:", baseline_ckpt)
print("num_variant_ckpts:", len(variant_ckpts))
print("num_models:", len(models))
for m in models:
    print(m["model_id"], "-> ckpt:", m["checkpoint_path"], "| rollouts:", m["scenario_rollouts_path"])


In [ ]:
# Step 4: Build simulation command plan (no metric ranking in this notebook)
from src.workflows import run_smart_eval_flow

flow_bundle = run_smart_eval_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    sync_smart_repo=True,
    models=models,
    strict_contract=False,
    require_metrics_binding=False,
    verify_checkpoint_hash=False,
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("first validate command:")
print(flow_bundle.models[0]["validate_cmd"])


In [ ]:
# Step 5: Optional simulation execution (resume-friendly)
RUN_SIM_ALL = _bool_env("WOSAC_RUN_SIM_ALL", False)
RUN_SIM_PENDING_ONLY = _bool_env("WOSAC_RUN_SIM_PENDING_ONLY", True)
MODEL_IDS_TO_RUN = [v.strip() for v in os.environ.get("WOSAC_MODEL_IDS_TO_RUN", "").split(",") if v.strip()]

def _rollout_path_for_model(model_id: str) -> Path:
    row = next((m for m in models if str(m.get("model_id")) == str(model_id)), {}) or {}
    return Path(str(row.get("scenario_rollouts_path", "")).strip()).expanduser()

def _pending(model_id: str) -> bool:
    p = _rollout_path_for_model(model_id)
    return (not p.exists()) or (p.stat().st_size == 0)

if MODEL_IDS_TO_RUN:
    selected = [m for m in flow_bundle.models if m["model_id"] in set(MODEL_IDS_TO_RUN)]
elif RUN_SIM_ALL:
    selected = list(flow_bundle.models)
else:
    selected = [m for m in flow_bundle.models if _pending(m["model_id"])] if RUN_SIM_PENDING_ONLY else []

if RUN_SIM_PENDING_ONLY:
    selected = [m for m in selected if _pending(m["model_id"])]

print("RUN_SIM_ALL:", RUN_SIM_ALL)
print("RUN_SIM_PENDING_ONLY:", RUN_SIM_PENDING_ONLY)
print("MODEL_IDS_TO_RUN:", MODEL_IDS_TO_RUN)
print("selected_models:", [m["model_id"] for m in selected])

for idx, model in enumerate(selected, start=1):
    cmd = model["validate_cmd"]
    print(f"[simulate {idx}/{len(selected)}] {model['model_id']} -> {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("Simulation execution block done.")


In [ ]:
# Step 6: Write per-model simulation manifests + run artifact (Drive)
from src.workflows import sha256_file, write_simulation_manifest

SIM_MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
manifest_rows = []

for model in flow_bundle.models:
    model_id = str(model.get("model_id", "")).strip()
    ckpt_path = str(model.get("checkpoint_path", "")).strip()
    scenario_rollouts_path = str((next((m for m in models if m.get("model_id") == model_id), {}) or {}).get("scenario_rollouts_path", "")).strip()
    manifest_path = SIM_MANIFESTS_DIR / f"{model_id}_simulation_manifest.json"
    manifest = write_simulation_manifest(
        manifest_path,
        {
            "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "run_tag": RUN_TAG,
            "run_name": RUN_NAME,
            "run_prefix": RUN_PREFIX,
            "model_id": model_id,
            "checkpoint_path": ckpt_path,
            "checkpoint_sha256": sha256_file(ckpt_path),
            "validate_cmd": str(model.get("validate_cmd", "")),
            "scenario_rollouts_path": scenario_rollouts_path,
            "scenario_proto_path": SCENARIO_PROTO_PATH,
            "scenario_proto_dir": SCENARIO_PROTO_DIR,
            "scenario_tfrecords": SCENARIO_TFRECORDS,
            "scenario_set_id": SCENARIO_SET_ID,
            "scenario_set_hash": SCENARIO_SET_HASH,
            "evaluator_id": EVALUATOR_ID,
            "metrics_config_hash": METRICS_CONFIG_HASH,
            "n_rollouts": int(RUN.get("n_rollouts_per_scenario", 32)),
            "num_history_seconds": int(RUN.get("num_history_seconds", 1)),
            "num_future_seconds": int(RUN.get("num_future_seconds", 8)),
            "seed": int(SIM_SEED),
            "smart_repo_rev": str((flow_bundle.summary.get("smart_repo_sync", {}) or {}).get("repo_rev", "unknown")),
            "source_notebook": "smart_rollout_simulation_colab.ipynb",
        },
    )
    manifest_rows.append({
        "model_id": model_id,
        "manifest_json": str(manifest_path),
        "manifest_sha256": manifest.get("manifest_sha256", ""),
        "checkpoint_sha256": manifest.get("checkpoint_sha256", ""),
        "scenario_rollouts_path": scenario_rollouts_path,
    })

payload = {
    "run_id": "smart_rollout_simulation_v0",
    "run_tag": RUN_TAG,
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "simulation_manifests_dir": str(SIM_MANIFESTS_DIR),
    "models": manifest_rows,
    "flow_summary": flow_bundle.summary,
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_path = RUN_OUTPUT_DIR / "smart_rollout_simulation_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("drive_path:", drive_path)
print("num_manifests:", len(manifest_rows))
